# 练习：不使用 SimpleSequentialChain 的链式调用

目标：在不使用 `SimpleSequentialChain` 的情况下，完成多步链式生成。

任务流程：`城市 -> 美食 -> 简介 -> 文案 -> 标题`。

In [ ]:
from llm_config import build_chat_openai

llm = build_chat_openai(temperature=0, request_timeout=120, max_tokens=1024)

## 练习要求

1. 使用 `prompt | llm | StrOutputParser()` 定义每一步；
2. 通过 Python 代码手动把上一步输出传给下一步；
3. 最终打印每一步结果，便于观察中间产物；
4. 不使用 `SimpleSequentialChain`。

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import PromptTemplate

# Step 1: 城市 -> 美食
cate_prompt = PromptTemplate.from_template(
    """
你是一位美食博主。
根据城市：{city}，推荐一种最有代表性的当地美食。
要求：只输出美食名称，不要解释，不要标点。
"""
)

# Step 2: 美食 -> 简介
intro_prompt = PromptTemplate.from_template(
    """
你是一位美食博主。
根据美食：{cate}，写一句简短介绍，突出特点和口感。
要求：只输出1句话，20-35字，不要分点。
"""
)

# Step 3: 简介 -> 种草文案
rewrite_prompt = PromptTemplate.from_template(
    """
你是一位小红书美食博主。
将下面这句介绍改写成有食欲、自然、像朋友分享的种草文案：
{intro}
要求：
1) 2-3句话；
2) 60-100字；
3) 只输出文案正文，不要标题，不要标签。
"""
)

# Step 4: 文案 -> 标题
title_prompt = PromptTemplate.from_template(
    """
请基于以下文案生成3个吸引人的中文标题：
{content}
要求：
1) 每个标题10-18字；
2) 不要使用emoji；
3) 严格按以下格式输出三行：
标题1：...
标题2：...
标题3：...
"""
)

# 组装每个 step（Runnable）
parser = StrOutputParser()
cate_step = cate_prompt | llm | parser
intro_step = intro_prompt | llm | parser
rewrite_step = rewrite_prompt | llm | parser
title_step = title_prompt | llm | parser

# 手动链式调用：显式传递中间结果
city = "成都"
cate = cate_step.invoke({"city": city}).strip()
intro = intro_step.invoke({"cate": cate}).strip()
content = rewrite_step.invoke({"intro": intro}).strip()
titles = title_step.invoke({"content": content}).strip()

print("===== 手动链式调用结果（无 SimpleSequentialChain）=====")
print(f"城市：{city}")
print(f"美食：{cate}")
print(f"简介：{intro}")
print(f"文案：{content}")
print(titles)